[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/04_export_onnx.ipynb)

# Notebook 04 — ONNX Export & HuggingFace Hub

Merges the LoRA adapter into LFM2-350M, exports to ONNX with int4 quantization, and publishes to HuggingFace Hub for browser inference.

**Prerequisites:** Run `02_finetune.ipynb` first to publish the LoRA adapter.

**Requirements:** T4 GPU, `GITHUB_TOKEN` and `HF_TOKEN` Colab secrets.

**Outputs:**
- `signal38/lfm2-nk-risk-onnx` — ONNX int4 model on HuggingFace Hub

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('04_export_onnx', requirements_path=str(REPO / 'requirements.txt'))
# Runtime may restart after this cell.

In [ ]:
import subprocess, sys, os, json, re
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook, require_local_adapter
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)

# Verify prerequisites
require_local_adapter(REPO)  # raises if adapter not published

print('Prerequisites satisfied.')

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(PATHS['adapter_dir']),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Merge LoRA weights into base model (produces a standard HF model)
merged_dir = PATHS['models_dir'] / 'lfm2-nk-risk' / 'merged'
merged_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained_merged(
    str(merged_dir),
    tokenizer,
    save_method='merged_16bit',
)
print(f'Merged model saved to {merged_dir}')
print('Files:', [f.name for f in merged_dir.iterdir()])

In [ ]:
import subprocess, sys

onnx_dir = PATHS['models_dir'] / 'lfm2-nk-risk' / 'onnx'
onnx_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run([
    sys.executable, '-m', 'optimum.exporters.onnx',
    '--model', str(merged_dir),
    '--task', 'text-generation-with-past',
    '--device', 'cuda',
    '--dtype', 'fp16',
    str(onnx_dir),
], capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('ONNX export failed — see stderr above')

print(f'ONNX export complete. Files: {[f.name for f in onnx_dir.iterdir()]}')

In [ ]:
from optimum.onnxruntime import ORTModelForCausalLM
from optimum.onnxruntime.configuration import AutoQuantizationConfig

quantized_dir = PATHS['models_dir'] / 'lfm2-nk-risk' / 'onnx_int4'
quantized_dir.mkdir(parents=True, exist_ok=True)

# Load the exported ONNX model
ort_model = ORTModelForCausalLM.from_pretrained(str(onnx_dir), export=False)

# Apply int4 quantization
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
from optimum.onnxruntime import ORTQuantizer
quantizer = ORTQuantizer.from_pretrained(ort_model)
quantizer.quantize(
    save_dir=str(quantized_dir),
    quantization_config=qconfig,
)

tokenizer.save_pretrained(str(quantized_dir))
print(f'int4 quantized model saved to {quantized_dir}')
print('Files:', [f.name for f in quantized_dir.iterdir()])

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Missing HF_TOKEN Colab secret. Add it via the key icon in the Colab left sidebar.')

login(token=hf_token)
api = HfApi()

REPO_ID = 'signal38/lfm2-nk-risk-onnx'

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)
    print(f'Hub repo ready: {REPO_ID}')
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload model files
api.upload_folder(
    folder_path=str(quantized_dir),
    repo_id=REPO_ID,
    repo_type='model',
    commit_message='Add LFM2-350M int4 ONNX NK risk model',
)
print(f'Pushed to https://huggingface.co/{REPO_ID}')

In [ ]:
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

# Load back from Hub to verify
ort_model_hub = ORTModelForCausalLM.from_pretrained(REPO_ID, export=False)
tokenizer_hub  = AutoTokenizer.from_pretrained(REPO_ID)

test_prompt = "Analyze North Korean military activity: high Goldstein negativity, missile-adjacent events. Respond in JSON."
inputs = tokenizer_hub(test_prompt, return_tensors='pt')
with torch.no_grad():
    out = ort_model_hub.generate(**inputs, max_new_tokens=64)
print(tokenizer_hub.decode(out[0], skip_special_tokens=True)[:500])
print('Smoke test passed.')